# Données OSM et SITG autour des arrêts TP

Objectif : préparer des cercles concentriques autour des arrêts de transport public, récupérer les équipements et services OSM, conserver les horaires d'ouverture, distinguer jour / nuit / 24h, puis agréger les accidents SITG depuis 2023.

Sources utilisées :
- OSM via Overpass / OSMnx pour les équipements, commerces et services.
- SITG `TPG_ARRETS` pour les arrêts TPG si aucun fichier local n'est fourni.
- SITG `OTC_ACCIDENTS` pour les accidents de circulation depuis 2023.

Sorties principales dans `../Data/processed/` : buffers/rings, POI OSM affectés aux rings, métriques OSM, accidents affectés aux rings, métriques accidents.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import re
import time
from typing import Iterable

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import LineString, MultiLineString, MultiPoint, MultiPolygon, Point, Polygon
from shapely.ops import unary_union
from tqdm.auto import tqdm

CRS_WGS84 = "EPSG:4326"
CRS_METRIC = "EPSG:2056"  # CH1903+ / LV95, mètres

NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebooks" else Path("..").resolve()
DATA_DIR = BASE_DIR / "Data"
INPUT_DIR = DATA_DIR / "input"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

for directory in [INPUT_DIR, RAW_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RING_RADII_M = [250, 500, 750, 1000]
ACCIDENTS_MIN_YEAR = 2023

TPG_ARRETS_URL = "https://vector.sitg.ge.ch/arcgis/rest/services/TPG_ARRETS/FeatureServer"
OTC_ACCIDENTS_URL = "https://vector.sitg.ge.ch/arcgis/rest/services/OTC_ACCIDENTS/FeatureServer"

ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.timeout = 240

print(BASE_DIR.resolve())

## Configuration OSM

La sélection reste volontairement large pour démarrer : équipements publics, santé, commerces, restauration, loisirs, tourisme, bureaux/services, transport et sécurité. Les tags bruts sont conservés pour permettre de raffiner la taxonomie ensuite.

In [ ]:
OSM_TAGS = {
    "amenity": True,
    "shop": True,
    "leisure": True,
    "tourism": True,
    "healthcare": True,
    "office": True,
    "craft": True,
    "public_transport": True,
    "railway": ["station", "halt", "tram_stop", "subway_entrance"],
    "emergency": True,
}

CATEGORY_TAG_PRIORITY = [
    "amenity",
    "shop",
    "healthcare",
    "leisure",
    "tourism",
    "office",
    "craft",
    "public_transport",
    "railway",
    "emergency",
]

DAY_START_HOUR = 5
NIGHT_START_HOUR = 22
NIGHT_END_HOUR = 5

def first_present_tag(row: pd.Series, tags: Iterable[str] = CATEGORY_TAG_PRIORITY) -> str:
    for tag in tags:
        value = row.get(tag)
        if pd.notna(value):
            return f"{tag}:{value}"
    return "unknown"

def safe_column_name(value: object) -> str:
    text = re.sub(r"[^0-9a-zA-Z]+", "_", str(value).lower()).strip("_")
    return text or "unknown"

def exportable_osm_subset(features: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    keep = [
        "osm_type", "osmid", "name", "osm_category", "opening_hours_raw",
        "open_day", "open_night", "open_24h", "geometry",
        *CATEGORY_TAG_PRIORITY,
    ]
    existing = [col for col in keep if col in features.columns]
    subset = features[existing].copy()
    subset.columns = [safe_column_name(col) if col != "geometry" else col for col in subset.columns]
    return gpd.GeoDataFrame(subset, geometry="geometry", crs=features.crs)

def parse_hour(value: str) -> float | None:
    match = re.match(r"^(\d{1,2})(?::(\d{2}))?$", value.strip())
    if not match:
        return None
    hour = int(match.group(1))
    minute = int(match.group(2) or 0)
    if hour > 24 or minute > 59:
        return None
    return min(24.0, hour + minute / 60)

def intervals_from_opening_hours(opening_hours: object) -> list[tuple[float, float]]:
    if not isinstance(opening_hours, str) or not opening_hours.strip():
        return []
    text = opening_hours.lower()
    if "24/7" in text:
        return [(0, 24)]

    intervals: list[tuple[float, float]] = []
    for start_raw, end_raw in re.findall(r"(\d{1,2}(?::\d{2})?)\s*-\s*(\d{1,2}(?::\d{2})?)", text):
        start = parse_hour(start_raw)
        end = parse_hour(end_raw)
        if start is None or end is None:
            continue
        if end < start:
            intervals.extend([(start, 24), (0, end)])
        else:
            intervals.append((start, end))
    return intervals

def overlaps_window(intervals: list[tuple[float, float]], start: float, end: float) -> bool | pd.NA:
    if not intervals:
        return pd.NA
    windows = [(start, 24), (0, end)] if end < start else [(start, end)]
    return any(max(a, c) < min(b, d) for a, b in intervals for c, d in windows)

def is_24h(opening_hours: object) -> bool | pd.NA:
    if not isinstance(opening_hours, str) or not opening_hours.strip():
        return pd.NA
    return "24/7" in opening_hours.lower()

def is_open_at_night(opening_hours: object) -> bool | pd.NA:
    return overlaps_window(intervals_from_opening_hours(opening_hours), NIGHT_START_HOUR, NIGHT_END_HOUR)

def is_open_during_day(opening_hours: object) -> bool | pd.NA:
    return overlaps_window(intervals_from_opening_hours(opening_hours), DAY_START_HOUR, NIGHT_START_HOUR)

## Fonctions SITG / ArcGIS REST

In [ ]:
def arcgis_query(feature_server_url: str, where: str = "1=1", out_fields: str = "*", result_record_count: int = 2000) -> dict:
    all_features = []
    offset = 0
    metadata: dict | None = None

    while True:
        params = {
            "where": where,
            "outFields": out_fields,
            "outSR": 4326,
            "f": "json",
            "returnGeometry": "true",
            "resultRecordCount": result_record_count,
            "resultOffset": offset,
        }
        response = requests.get(f"{feature_server_url}/0/query", params=params, timeout=90)
        response.raise_for_status()
        data = response.json()
        if "error" in data:
            raise RuntimeError(data["error"])
        metadata = metadata or {key: value for key, value in data.items() if key != "features"}
        features = data.get("features", [])
        all_features.extend(features)
        if not data.get("exceededTransferLimit") or not features:
            break
        offset += len(features)
        time.sleep(0.2)

    assert metadata is not None
    metadata["features"] = all_features
    return metadata

def esri_geometry_to_shapely(geometry: dict | None):
    if not geometry:
        return None
    if "x" in geometry and "y" in geometry:
        return Point(geometry["x"], geometry["y"])
    if "points" in geometry:
        points = [Point(x, y) for x, y in geometry["points"]]
        return MultiPoint(points) if len(points) > 1 else points[0]
    if "paths" in geometry:
        lines = [LineString(path) for path in geometry["paths"] if len(path) >= 2]
        return MultiLineString(lines) if len(lines) > 1 else (lines[0] if lines else None)
    if "rings" in geometry:
        polygons = [Polygon(ring) for ring in geometry["rings"] if len(ring) >= 4]
        polygons = [polygon for polygon in polygons if polygon.is_valid and not polygon.is_empty]
        return MultiPolygon(polygons) if len(polygons) > 1 else (polygons[0] if polygons else None)
    return None

def arcgis_json_to_gdf(data: dict, crs: str = CRS_WGS84) -> gpd.GeoDataFrame:
    rows = []
    for index, feature in enumerate(data.get("features", [])):
        attrs = feature.get("attributes", {}) or {}
        geom = esri_geometry_to_shapely(feature.get("geometry"))
        if geom is None:
            continue
        rows.append({**attrs, "_feature_index": index, "geometry": geom})
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=crs)

def average_multipoint(geometry):
    if isinstance(geometry, Point):
        return geometry
    if isinstance(geometry, MultiPoint):
        xs = [point.x for point in geometry.geoms]
        ys = [point.y for point in geometry.geoms]
        return Point(float(np.mean(xs)), float(np.mean(ys)))
    return geometry.representative_point()

## Arrêts TP

Si `Data/input/arrets_tp.csv` existe, il est utilisé. Sinon le notebook récupère les arrêts TPG depuis SITG et ajoute un référentiel minimal Léman Express.

In [ ]:
LEMAN_EXPRESS_STATIONS = [
    ("leman-8501023", "Coppet", 6.187806, 46.317405, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501015", "Tannay", 6.181097, 46.307659, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501014", "Mies", 6.167641, 46.298503, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501013", "Pont-Céard", 6.162762, 46.286824, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501022", "Versoix", 6.165718, 46.279741, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501012", "Creux-de-Genthod", 6.161292, 46.263776, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501021", "Genthod-Bellevue", 6.153947, 46.256747, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501011", "Les Tuileries", 6.147312, 46.249960, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501020", "Chambésy", 6.147279, 46.240954, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8516283", "Genève-Sécheron", 6.144542, 46.222452, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8501008", "Genève", 6.142435, 46.210228, "Léman Express", "L1,L2,L3,L4,L5,L6"),
    ("leman-8516155", "Lancy-Pont-Rouge", 6.124929, 46.185960, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8517142", "Lancy-Bachet", 6.129340, 46.174342, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8516272", "Genève-Champel", 6.153473, 46.192208, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8516273", "Genève-Eaux-Vives", 6.166549, 46.201461, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8516274", "Chêne-Bourg", 6.197316, 46.196612, "Léman Express", "L1,L2,L3,L4"),
    ("leman-8774549", "Annemasse", 6.236381, 46.199355, "Léman Express", "L1,L2,L3,L4"),
]

def slugify(value: str) -> str:
    return re.sub(r"(^-|-$)", "", re.sub(r"[^a-z0-9]+", "-", value.lower())).strip("-")

def fetch_tpg_stops() -> gpd.GeoDataFrame:
    data = arcgis_query(
        TPG_ARRETS_URL,
        where="1=1",
        out_fields="NOM_ARRET,LIGNE,DIRECTION,OBJECTID",
        result_record_count=4000,
    )
    raw = arcgis_json_to_gdf(data)
    raw.to_file(RAW_DIR / "tpg_arrets_raw.geojson", driver="GeoJSON")
    raw["stop_name"] = raw["NOM_ARRET"].astype(str)
    raw["line"] = raw["LIGNE"].astype(str).str.replace(r"^0", "", regex=True)
    raw["point"] = raw.geometry.apply(average_multipoint)

    rows = []
    for name, group in raw.groupby("stop_name"):
        points = gpd.GeoSeries(group["point"], crs=CRS_WGS84).to_crs(CRS_METRIC)
        center = Point(points.x.mean(), points.y.mean())
        center_wgs = gpd.GeoSeries([center], crs=CRS_METRIC).to_crs(CRS_WGS84).iloc[0]
        rows.append({
            "stop_id": f"tpg-{slugify(name)}",
            "name": name,
            "network": "TPG",
            "lines": ",".join(sorted(group["line"].dropna().unique(), key=str)),
            "geometry": center_wgs,
        })
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_WGS84)

def load_or_fetch_stations() -> gpd.GeoDataFrame:
    local_file = INPUT_DIR / "arrets_tp.csv"
    if local_file.exists():
        df = pd.read_csv(local_file)
        required = {"stop_id", "name", "lon", "lat", "network", "lines"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"Colonnes manquantes dans {local_file}: {sorted(missing)}")
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs=CRS_WGS84)

    tpg = fetch_tpg_stops()
    leman = pd.DataFrame(LEMAN_EXPRESS_STATIONS, columns=["stop_id", "name", "lon", "lat", "network", "lines"])
    leman = gpd.GeoDataFrame(leman, geometry=gpd.points_from_xy(leman["lon"], leman["lat"]), crs=CRS_WGS84)
    stations = pd.concat([tpg, leman], ignore_index=True)
    stations["lon"] = stations.geometry.x
    stations["lat"] = stations.geometry.y
    return gpd.GeoDataFrame(stations, geometry="geometry", crs=CRS_WGS84)

stations = load_or_fetch_stations()
stations.to_file(PROCESSED_DIR / "tp_stations.geojson", driver="GeoJSON")
stations.drop(columns="geometry").to_csv(PROCESSED_DIR / "tp_stations.csv", index=False)
stations.head()

## Cercles concentriques autour des arrêts

In [ ]:
def make_station_rings(stations_gdf: gpd.GeoDataFrame, radii_m: list[int]) -> gpd.GeoDataFrame:
    metric = stations_gdf.to_crs(CRS_METRIC)
    rows = []
    for _, stop in metric.iterrows():
        previous = None
        previous_radius = 0
        for radius in radii_m:
            buffer = stop.geometry.buffer(radius)
            ring = buffer if previous is None else buffer.difference(previous)
            rows.append({
                "stop_id": stop["stop_id"],
                "stop_name": stop["name"],
                "network": stop.get("network"),
                "lines": stop.get("lines"),
                "ring_from_m": previous_radius,
                "ring_to_m": radius,
                "ring_label": f"{previous_radius}-{radius}m",
                "geometry": ring,
            })
            previous = buffer
            previous_radius = radius
    rings = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_METRIC)
    rings["ring_area_m2"] = rings.area
    return rings

station_rings = make_station_rings(stations, RING_RADII_M)
station_rings.to_file(PROCESSED_DIR / "station_concentric_rings.gpkg", layer="rings", driver="GPKG")
station_rings.to_crs(CRS_WGS84).to_file(PROCESSED_DIR / "station_concentric_rings.geojson", driver="GeoJSON")
station_rings.head()

## Collecte OSM et affectation aux rings

In [ ]:
def fetch_osm_features_for_rings(rings: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    area = unary_union(rings.to_crs(CRS_WGS84).geometry)
    print("Requête Overpass sur l'emprise des buffers...")
    features = ox.features_from_polygon(area, OSM_TAGS)
    features = features.reset_index()
    if "element_type" in features.columns and "osm_type" not in features.columns:
        features = features.rename(columns={"element_type": "osm_type"})
    for required_col in ["osm_type", "osmid", "name", "opening_hours"]:
        if required_col not in features.columns:
            features[required_col] = pd.NA
    features = gpd.GeoDataFrame(features, geometry="geometry", crs=CRS_WGS84)
    features = features[features.geometry.notna() & ~features.geometry.is_empty].copy()
    features["osm_category"] = features.apply(first_present_tag, axis=1)
    features["opening_hours_raw"] = features.get("opening_hours")
    features["open_day"] = features["opening_hours_raw"].apply(is_open_during_day)
    features["open_night"] = features["opening_hours_raw"].apply(is_open_at_night)
    features["open_24h"] = features["opening_hours_raw"].apply(is_24h)
    return features

osm_features = fetch_osm_features_for_rings(station_rings)
exportable_osm_subset(osm_features).to_file(RAW_DIR / "osm_station_buffer_features.gpkg", layer="osm_features", driver="GPKG")
osm_features[["osm_type", "osmid", "name", "osm_category", "opening_hours_raw", "open_day", "open_night", "open_24h"]].head()

In [ ]:
def representative_points(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = gdf.copy()
    out["original_geometry_type"] = out.geometry.geom_type
    out["geometry"] = out.geometry.apply(lambda geom: geom if isinstance(geom, Point) else geom.representative_point())
    return gpd.GeoDataFrame(out, geometry="geometry", crs=gdf.crs)

def assign_features_to_rings(features: gpd.GeoDataFrame, rings: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    points = representative_points(features).to_crs(CRS_METRIC)
    ring_cols = ["stop_id", "stop_name", "network", "lines", "ring_from_m", "ring_to_m", "ring_label", "geometry"]
    ring_attrs = rings[ring_cols].rename(columns={col: f"ring_{col}" for col in ring_cols if col != "geometry"})
    joined = gpd.sjoin(points, ring_attrs, how="inner", predicate="within")
    joined = joined.drop(columns=["index_right"], errors="ignore")
    joined = joined.rename(columns={f"ring_{col}": col for col in ring_cols if col != "geometry"})
    return joined

osm_by_ring = assign_features_to_rings(osm_features, station_rings)
exportable_osm_subset(osm_by_ring).assign(
    stop_id=osm_by_ring["stop_id"],
    stop_name=osm_by_ring["stop_name"],
    network=osm_by_ring["network"],
    lines=osm_by_ring["lines"],
    ring_label=osm_by_ring["ring_label"],
    ring_from_m=osm_by_ring["ring_from_m"],
    ring_to_m=osm_by_ring["ring_to_m"],
).to_file(PROCESSED_DIR / "osm_features_by_station_ring.gpkg", layer="osm_by_ring", driver="GPKG")

osm_metrics = (
    osm_by_ring.assign(
        has_opening_hours=osm_by_ring["opening_hours_raw"].notna(),
        open_day_num=osm_by_ring["open_day"].fillna(False).astype(bool),
        open_night_num=osm_by_ring["open_night"].fillna(False).astype(bool),
        open_24h_num=osm_by_ring["open_24h"].fillna(False).astype(bool),
    )
    .groupby(["stop_id", "stop_name", "network", "lines", "ring_label", "ring_from_m", "ring_to_m"], dropna=False)
    .agg(
        osm_features_count=("osmid", "count"),
        osm_categories_count=("osm_category", "nunique"),
        features_with_opening_hours=("has_opening_hours", "sum"),
        open_day_count=("open_day_num", "sum"),
        open_night_count=("open_night_num", "sum"),
        open_24h_count=("open_24h_num", "sum"),
    )
    .reset_index()
)

category_counts = (
    osm_by_ring.groupby(["stop_id", "ring_label", "osm_category"], dropna=False)
    .size()
    .reset_index(name="count")
)

osm_metrics.to_csv(PROCESSED_DIR / "osm_metrics_by_station_ring.csv", index=False)
category_counts.to_csv(PROCESSED_DIR / "osm_category_counts_by_station_ring.csv", index=False)

station_ring_equipment_wide = (
    category_counts.assign(equipment_col=category_counts["osm_category"].apply(safe_column_name))
    .pivot_table(index=["stop_id", "ring_label"], columns="equipment_col", values="count", aggfunc="sum", fill_value=0)
    .reset_index()
)
station_ring_equipment_wide = station_ring_equipment_wide.merge(
    osm_metrics,
    on=["stop_id", "ring_label"],
    how="left",
)
station_ring_equipment_wide.to_csv(PROCESSED_DIR / "station_ring_equipment_wide.csv", index=False)

wide_parts = []
for ring_label, group in station_ring_equipment_wide.groupby("ring_label"):
    prefix = f"r{str(ring_label).split('-')[-1].replace('m', '')}_"
    part = group.drop(columns=["ring_label"])
    part = part.rename(columns={col: f"{prefix}{col}" for col in part.columns if col != "stop_id"})
    wide_parts.append(part)

station_equipment_wide = stations.drop(columns="geometry").copy()
for part in wide_parts:
    station_equipment_wide = station_equipment_wide.merge(part, on="stop_id", how="left")
station_equipment_wide = station_equipment_wide.fillna(0)
station_equipment_wide.to_csv(PROCESSED_DIR / "station_equipment_wide.csv", index=False)
osm_metrics.head()

## Accidents SITG depuis 2023

La couche `OTC_ACCIDENTS` contient les accidents depuis 2010. Ici on ne conserve que `ANNEE >= 2023` pour l'analyse autour des arrêts.

In [ ]:
accidents_data = arcgis_query(
    OTC_ACCIDENTS_URL,
    where=f"ANNEE >= {ACCIDENTS_MIN_YEAR}",
    out_fields="ID_ACCIDENT,DATE_,HEURE,ANNEE,JOUR,GROUPE_ACCIDENT,TYPE,CAUSE,COMMUNE,CONDITIONS_LUMINEUSES,CONDITIONS_METEO,CONSEQUENCES,NB_BLESSES_LEGERS,NB_BLESSES_GRAVES,NB_TUES,NB_PIETONS,NB_BICYCLETTES,NB_VAE_25,NB_VAE_45,NB_TC,TOTAL_PERSONNES",
    result_record_count=2000,
)
accidents = arcgis_json_to_gdf(accidents_data)
accidents["DATE_UTC"] = pd.to_datetime(accidents.get("DATE_"), unit="ms", errors="coerce")
accidents["hour_raw"] = accidents.get("HEURE")
accidents.to_file(RAW_DIR / "sitg_accidents_since_2023.geojson", driver="GeoJSON")

accidents_by_ring = gpd.sjoin(accidents.to_crs(CRS_METRIC), station_rings, how="inner", predicate="within")
accidents_by_ring = accidents_by_ring.drop(columns=["index_right"], errors="ignore")
accidents_by_ring.to_file(PROCESSED_DIR / "accidents_by_station_ring.gpkg", layer="accidents_by_ring", driver="GPKG")

for col in ["NB_BLESSES_LEGERS", "NB_BLESSES_GRAVES", "NB_TUES", "NB_PIETONS", "NB_BICYCLETTES", "NB_VAE_25", "NB_VAE_45", "NB_TC", "TOTAL_PERSONNES"]:
    accidents_by_ring[col] = pd.to_numeric(accidents_by_ring.get(col), errors="coerce").fillna(0)

accident_metrics = (
    accidents_by_ring.assign(
        severe_or_fatal=lambda df: (df["NB_BLESSES_GRAVES"] + df["NB_TUES"]) > 0,
        active_modes_involved=lambda df: (df["NB_PIETONS"] + df["NB_BICYCLETTES"] + df["NB_VAE_25"] + df["NB_VAE_45"]) > 0,
    )
    .groupby(["stop_id", "stop_name", "network", "lines", "ring_label", "ring_from_m", "ring_to_m"], dropna=False)
    .agg(
        accidents_count=("ID_ACCIDENT", "count"),
        severe_or_fatal_count=("severe_or_fatal", "sum"),
        active_modes_accidents_count=("active_modes_involved", "sum"),
        injured_light=("NB_BLESSES_LEGERS", "sum"),
        injured_serious=("NB_BLESSES_GRAVES", "sum"),
        killed=("NB_TUES", "sum"),
        pedestrians_involved=("NB_PIETONS", "sum"),
        bicycles_involved=("NB_BICYCLETTES", "sum"),
        ebikes_25_involved=("NB_VAE_25", "sum"),
        ebikes_45_involved=("NB_VAE_45", "sum"),
        public_transport_involved=("NB_TC", "sum"),
    )
    .reset_index()
)

accident_metrics.to_csv(PROCESSED_DIR / "accident_metrics_by_station_ring.csv", index=False)
accident_metrics.head()

## Emplacement pour l'indice de précarité

Le notebook prépare un emplacement stable. Lorsque la donnée sera disponible dans `Data/input/`, elle pourra être jointe aux rings par intersection ou par pondération surfacique.

In [ ]:
precarity_candidates = [
    INPUT_DIR / "indice_precarite.geojson",
    INPUT_DIR / "indice_precarite.gpkg",
    INPUT_DIR / "indice_precarite.csv",
]

precarity_path = next((path for path in precarity_candidates if path.exists()), None)

if precarity_path is None:
    template = pd.DataFrame(columns=["zone_id", "indice_precarite", "source", "year", "notes"])
    template.to_csv(INPUT_DIR / "indice_precarite_template.csv", index=False)
    print("Aucune donnée de précarité trouvée. Template créé dans Data/input/indice_precarite_template.csv")
else:
    print(f"Donnée de précarité détectée : {precarity_path}")
    if precarity_path.suffix.lower() == ".csv":
        precarity = pd.read_csv(precarity_path)
    else:
        precarity = gpd.read_file(precarity_path).to_crs(CRS_METRIC)
    display(precarity.head())

## Exports générés

- `Data/processed/tp_stations.geojson`
- `Data/processed/station_concentric_rings.gpkg`
- `Data/raw/osm_station_buffer_features.gpkg`
- `Data/processed/osm_features_by_station_ring.gpkg`
- `Data/processed/osm_metrics_by_station_ring.csv`
- `Data/processed/osm_category_counts_by_station_ring.csv`
- `Data/raw/sitg_accidents_since_2023.geojson`
- `Data/processed/accidents_by_station_ring.gpkg`
- `Data/processed/accident_metrics_by_station_ring.csv`